# Talent Intelligence & Skills Gap Analysis
## 03 — NLP Skill Extraction from Job Descriptions

**Scenario:** NorthStar Digital Services is fictional. All job descriptions and workforce data are synthetic.

### Objectives
1. Normalize free-text job descriptions.
2. Extract skills using the project skills taxonomy and aliases.
3. Map extracted terms to canonical skills.
4. Compare NLP-extracted skills with structured role-skill requirements.
5. Measure Precision, Recall, and F1 Score.
6. Save outputs for GitHub and later Power BI use.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)


### 1. Connect to the GitHub repository in Google Colab
Run this cell in a new Colab session.


In [ ]:
REPO_URL = 'https://github.com/mazaheriasad/talent-intelligence-skills-gap-analysis.git'
REPO_NAME = 'talent-intelligence-skills-gap-analysis'
REPO_DIR = Path('/content') / REPO_NAME

if not REPO_DIR.exists():
    !git clone {REPO_URL}

%cd /content/{REPO_NAME}


### 2. Load source data


In [ ]:
DATA_DIR = Path('data/raw')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

skills_taxonomy = pd.read_csv(DATA_DIR / 'skills_taxonomy.csv')
target_roles = pd.read_csv(DATA_DIR / 'target_roles.csv')
role_skills = pd.read_csv(DATA_DIR / 'role_skills.csv')

print('Taxonomy skills:', len(skills_taxonomy))
print('Target roles:', len(target_roles))
print('Structured role-skill requirements:', len(role_skills))


### 3. Build the canonical skill dictionary


In [ ]:
def split_aliases(value):
    if pd.isna(value):
        return []
    return [x.strip().lower() for x in str(value).split(';') if x.strip()]

skill_dictionary = {}
for _, row in skills_taxonomy.iterrows():
    canonical = row['CanonicalSkill']
    aliases = split_aliases(row['Aliases_For_NLP'])
    aliases.append(canonical.lower())
    skill_dictionary[canonical] = sorted(set(aliases), key=len, reverse=True)

list(skill_dictionary.items())[:5]


### 4. Normalize text and extract taxonomy-matched skills


In [ ]:
def normalize_text(text):
    text = str(text).lower()
    text = text.replace('&', ' and ')
    text = re.sub(r'[_/|]+', ' ', text)
    text = re.sub(r'[^a-z0-9+#.\-\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def phrase_present(alias, text):
    pattern = r'(?<![a-z0-9])' + re.escape(alias) + r'(?![a-z0-9])'
    return re.search(pattern, text) is not None

def extract_skills(text):
    normalized = normalize_text(text)
    found = []
    for canonical, aliases in skill_dictionary.items():
        for alias in aliases:
            normalized_alias = normalize_text(alias)
            if phrase_present(normalized_alias, normalized):
                found.append((canonical, alias))
                break
    return found


### 5. Extract skills from all target-role job descriptions


In [ ]:
rows = []
for _, role in target_roles.iterrows():
    for canonical, alias in extract_skills(role['JobDescription']):
        tx = skills_taxonomy.loc[skills_taxonomy['CanonicalSkill'].eq(canonical)].iloc[0]
        rows.append({
            'RoleID': role['RoleID'],
            'RoleName': role['RoleName'],
            'CanonicalSkill': canonical,
            'MatchedAlias': alias,
            'SkillCategory': tx['SkillCategory'],
            'SkillType': tx['SkillType']
        })

nlp_extracted = pd.DataFrame(rows)
nlp_extracted.head(20)


### 6. Extraction count by role


In [ ]:
extraction_counts = (
    nlp_extracted.groupby(['RoleID','RoleName'])
    .agg(ExtractedSkills=('CanonicalSkill','nunique'))
    .reset_index()
)
extraction_counts


### 7. Evaluate extraction against structured role requirements

- **TP:** extracted and expected
- **FP:** extracted but not expected
- **FN:** expected but not extracted


In [ ]:
evaluation_rows = []
comparison_rows = []

for _, role in target_roles[['RoleID','RoleName']].drop_duplicates().iterrows():
    role_id = role['RoleID']
    role_name = role['RoleName']
    expected = set(role_skills.loc[role_skills['RoleID'].eq(role_id), 'Skill'])
    extracted = set(nlp_extracted.loc[nlp_extracted['RoleID'].eq(role_id), 'CanonicalSkill'])
    tp = expected & extracted
    fp = extracted - expected
    fn = expected - extracted
    precision = len(tp) / (len(tp) + len(fp)) if (len(tp) + len(fp)) else 0
    recall = len(tp) / (len(tp) + len(fn)) if (len(tp) + len(fn)) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    evaluation_rows.append({
        'RoleID': role_id, 'RoleName': role_name,
        'ExpectedSkills': len(expected), 'ExtractedSkills': len(extracted),
        'TruePositives': len(tp), 'FalsePositives': len(fp), 'FalseNegatives': len(fn),
        'Precision': precision, 'Recall': recall, 'F1Score': f1
    })
    for skill in sorted(expected | extracted):
        status = 'True Positive' if skill in tp else ('False Positive' if skill in fp else 'False Negative')
        comparison_rows.append({
            'RoleID': role_id, 'RoleName': role_name, 'Skill': skill,
            'Expected': skill in expected, 'Extracted': skill in extracted,
            'EvaluationStatus': status
        })

nlp_evaluation = pd.DataFrame(evaluation_rows)
nlp_skill_comparison = pd.DataFrame(comparison_rows)
nlp_evaluation.round(3)


### 8. Overall NLP evaluation


In [ ]:
tp = nlp_evaluation['TruePositives'].sum()
fp = nlp_evaluation['FalsePositives'].sum()
fn = nlp_evaluation['FalseNegatives'].sum()
precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

overall_metrics = pd.DataFrame([{
    'TruePositives': tp, 'FalsePositives': fp, 'FalseNegatives': fn,
    'Precision': precision, 'Recall': recall, 'F1Score': f1
}])
overall_metrics.round(3)


### 9. F1 Score by target role


In [ ]:
plot_data = nlp_evaluation.sort_values('F1Score')
plt.figure(figsize=(10,6))
plt.barh(plot_data['RoleName'], plot_data['F1Score'])
plt.xlabel('F1 Score')
plt.ylabel('Target Role')
plt.title('NLP Skill Extraction Performance by Role')
plt.xlim(0,1)
plt.tight_layout()
plt.show()


### 10. Inspect missed or extra skills


In [ ]:
issues = nlp_skill_comparison[
    nlp_skill_comparison['EvaluationStatus'].ne('True Positive')
].copy()
issues.sort_values(['RoleID','EvaluationStatus','Skill'])


### 11. Role skill coverage summary


In [ ]:
role_coverage = nlp_evaluation[[
    'RoleID','RoleName','ExpectedSkills','ExtractedSkills','Precision','Recall','F1Score'
]].copy()
role_coverage['SkillCoveragePct'] = role_coverage['Recall'] * 100
role_coverage.sort_values('SkillCoveragePct', ascending=False).round(2)


### 12. Save outputs


In [ ]:
nlp_extracted.to_csv(OUTPUT_DIR / 'nlp_extracted_role_skills.csv', index=False)
nlp_evaluation.to_csv(OUTPUT_DIR / 'nlp_extraction_evaluation.csv', index=False)
nlp_skill_comparison.to_csv(OUTPUT_DIR / 'nlp_skill_comparison.csv', index=False)
overall_metrics.to_csv(OUTPUT_DIR / 'nlp_overall_metrics.csv', index=False)
role_coverage.to_csv(OUTPUT_DIR / 'nlp_role_skill_coverage.csv', index=False)

print('Saved NLP outputs to:', OUTPUT_DIR.resolve())


## Interpretation note
This is an interpretable taxonomy-based NLP baseline. A production version could add embeddings or an LLM, but the controlled skills taxonomy remains important for canonicalization, governance, and auditability.

## Next phase
Notebook 04 will combine **Role Readiness + Skill Gaps + Mobility Interest** to build internal talent pools and learning recommendations.
